# Data Cleaning

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [7]:
df = pd.read_csv("../data/shopee_hoya_master.csv")

print("Raw dataset shape:")
print(df.shape)

df.head()

Raw dataset shape:
(623, 11)


,product_name,variety,price,number_sold,location,rating,number_of_reviews,marketplace,scraping_date,product_url,shop_name
0,Hoya Cumigiana,Cumigiana,10000,5,Kab. Lumajang,4.90,497,Shopee,2026-06-11,https://shopee.co.id/product/348755541/8915296529,Ngundang anggrek
1,Hoya Lacunosa Rimbun Bunganya Wangi,Lacunosa,12000,5,Kab. Lumajang,4.93,307,Shopee,2026-06-11,https://shopee.co.id/product/348755541/6579385658,Ngundang anggrek
2,Tanaman Hoya Carnosa Tricolor Atau Crimsom Pri...,Carnosa,34500,3,Kab. Bandung Barat,4.98,84,Shopee,2026-06-11,https://shopee.co.id/product/104126944/6352344959,Shofiflora
3,Sekulen Cluster Hoya Love Varigata Bercabang,Love,27500,7,Kab. Bandung Barat,4.94,48,Shopee,2026-06-11,https://shopee.co.id/product/163019197/2871789...,Kaktus Store
4,Hoya kuning cerah,Kuning,20000,3,Kab. Lumajang,4.88,97,Shopee,2026-06-11,https://shopee.co.id/product/348755541/8787796573,Ngundang anggrek


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product_name       623 non-null    object 
 1   variety            623 non-null    object 
 2   price              623 non-null    int64  
 3   number_sold        623 non-null    int64  
 4   location           623 non-null    object 
 5   rating             623 non-null    float64
 6   number_of_reviews  623 non-null    int64  
 7   marketplace        623 non-null    object 
 8   scraping_date      623 non-null    object 
 9   product_url        623 non-null    object 
 10  shop_name          623 non-null    object 
dtypes: float64(1), int64(3), object(7)
memory usage: 53.7+ KB


In [9]:
df.isnull().sum()

product_name         0
variety              0
price                0
number_sold          0
location             0
rating               0
number_of_reviews    0
marketplace          0
scraping_date        0
product_url          0
shop_name            0
dtype: int64

In [10]:
df = df[
    df["product_name"]
    .str.lower()
    .str.contains("hoya", na=False)
]

print("Dataset shape after Hoya filtering:")
print(df.shape)

Dataset shape after Hoya filtering:
(623, 11)


In [12]:
# Remove Irrelevant Products
BAD_KEYWORDS = [

    "ank sale hoya",

    "artificial",
    "plastik",
    "fake",
    "palsu",

    "bibit",
    "seed",
    "seeds",

    "dischidia",

    "peperomia",
    "paperomia",

    "silver dollar",
    "xerosicyos",
    "xerocicyos",

    "paket",
    "mix",
    "lengkap",

    "mug",
    "benang",

    "lipstik",
    "lipstick",

    "hoyaplast",

    "mangkuk",
    "mangkok",

    "media",

    "tomolab",

    "pashmina",

    "sagu",

    "sukulen",

    "piring",

    "kering",

    "kaktus",

    "pinus",

    "tumbler",

    "batu",

    "pakis",

    "melati",

    "lavender",

    "mawar",

    "kancing",

    "monstera"
]

pattern = "|".join(BAD_KEYWORDS)

df = df[
    ~df["product_name"]
    .str.lower()
    .str.contains(pattern, na=False)
]

print("Dataset shape after irrelevant-product removal:")
print(df.shape)

Dataset shape after irrelevant-product removal:
(419, 11)


In [13]:
df["variety"] = (
    df["variety"]
    .astype(str)
    .str.strip()
)

In [14]:
# Fix Cornosa to Carnosa
df["variety"] = (
    df["variety"]
    .str.replace(
        r"(?i)cornosa",
        "Carnosa",
        regex=True
    )
)

In [15]:
# standardize capitalization
df["variety"] = (
    df["variety"]
    .str.title()
)

In [16]:
variety_mapping = {

    "Hoya Carnosa": "Carnosa",

    "Carnosa Splash": "Carnosa",

    "Compacta Variegata": "Compacta",

    "Krimsonqueen": "Krimson Queen",

    "Krimson Princess ": "Krimson Princess"

}

df["variety"] = df["variety"].replace(
    variety_mapping
)

In [17]:
df[
    df["variety"]
    .str.contains(
        "corn|carn",
        case=False,
        na=False
    )
]["variety"].value_counts()

variety
Carnosa    40
Name: count, dtype: int64

In [19]:
numeric_cols = [

    "price",

    "number_sold",

    "rating",

    "number_of_reviews"
]

for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

## Top Varieties

In [21]:
df["variety"].value_counts().head(30)

variety
Carnosa                  40
Tidak Teridentifikasi    39
Lacunosa                 21
Compacta                 17
Love                     15
Macrophylla              14
Diversifolia              6
Putih                     6
Curly                     6
Krohniana                 6
Costa                     5
Australis                 5
Pubicalyx                 5
Kerrii                    5
Aff                       4
Gantung                   4
Bunga                     4
Verticilata               3
Wayeti                    3
Imperialis                3
Callistophylla            3
Lakunosa                  3
Obovata                   3
Vitellina                 3
Waymaniae                 3
Curtisii                  3
Kerri                     3
Tanggamus                 3
Sunrise                   3
Archboldiana              3
Name: count, dtype: int64

In [22]:
print("="*40)
print("FINAL DATASET SUMMARY")
print("="*40)

print(
    f"Number of Listings: {len(df)}"
)

print(
    f"Number of Varieties: {df['variety'].nunique()}"
)

print(
    f"Number of Locations: {df['location'].nunique()}"
)

FINAL DATASET SUMMARY
Number of Listings: 419
Number of Varieties: 175
Number of Locations: 50


In [23]:
df.to_csv(
    "../data/shopee_hoya_clean.csv",
    index=False
)

print(
    "Dataset saved successfully."
)

Dataset saved successfully.
